# Compare controlled runs and check recommendations

Linden needs a defensible choice rather than five unrelated scores. The fixture holds data, split, scaling, solver and guard fixed while changing alpha.

Run all cells top to bottom. **Colab:** connect a fresh CPU / Python 3.12 runtime; the first cell clones the pinned GitHub source and installs its locked runtime dependencies. **Local:** run `uv sync --locked` from `module-a` and select that `.venv` interpreter. See [Colab and environment guidance](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/colab.md). Every run creates a new temporary tracking directory and prints its path; download evidence before ending a hosted runtime. No API key is needed. These are complete worked references; independent practices have separate starting states. IDs and timing vary.

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'eec754e0b1379fe8bb01e23543b92e036b4c6b62', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    for name, version in expected.items():
        if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version:
            raise RuntimeError(f'{name} is already loaded at another version. Delete/reconnect the runtime, then run setup first.')
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: eec754e0b1379fe8bb01e23543b92e036b4c6b62 state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-vruy7dfn


## Run the declared matrix
Autologging hooks model fitting. We disable automatic model logging inside the helper so the explicit named model remains unambiguous. The helper still logs the primary held-out metric itself.

In [2]:
ws=Workspace(state/'tracking')
records=run_matrix(ws,autolog=True)
selected_ids=[record['run_id'] for record in records]
(ws.path/'matrix-run-ids.json').write_text(json.dumps(selected_ids,indent=2))
all_rows=export_runs(ws)
rows=all_rows[all_rows.run_id.isin(selected_ids)].copy()
rows.to_csv(ws.path/'runs.csv',index=False)
display(rows[['run_id','alpha','range_guard','evaluation_partition','rmse_kwh','mae_kwh']].sort_values('rmse_kwh'))
assert len(rows)==5

2026/10/19 14:42:14 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 14:42:14 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:14 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:14 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 14:42:14 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026/10/19 14:42:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/19 14:42:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/19 14:42:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/19 14:42:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


,run_id,alpha,range_guard,evaluation_partition,rmse_kwh,mae_kwh
4,67dd70d163a24d3e8e4202b0917ae7ce,0.01,True,validation,1.434282,1.201891
3,62c595f83ee54e4a882a3c75d412be73,0.1,True,validation,1.434794,1.203859
2,9d73c3b0ecce4944bf056ca4fb20f146,1.0,True,validation,1.443004,1.223427
1,84d68ae424c5495eb16e8fb3e08dd34f,10.0,True,validation,1.766063,1.437359
0,069962b2a74647f6a857f94dbcb9e373,100.0,True,validation,6.715349,5.680055


## Inspect raw MLflow evidence
Auto-generated training metrics are useful diagnostics, not the declared decision metric. Inspect a raw run and artifact list instead of trusting a summary.

In [3]:
best=rows.sort_values('rmse_kwh').iloc[0]
raw=ws.client.get_run(best.run_id)
print('alpha:',best.alpha,'validation RMSE:',best.rmse_kwh)
print('Autolog parameter:',raw.data.params['ridge__alpha'])
print('Training metric:',raw.data.metrics['training_mean_squared_error'])
print([a.path for a in ws.client.list_artifacts(best.run_id,'reproduction')])
assert float(best.alpha)==0.01

alpha: 0.01 validation RMSE: 1.4342817684933908
Autolog parameter: 0.01
Training metric: 2.0040508496474954
['reproduction/config.json', 'reproduction/environment.txt', 'reproduction/manifest.json', 'reproduction/source.tar.gz']


## Validate an assistant recommendation
Export `runs.csv` to your institution-approved assistant only if its data policy permits. Ask it to name run IDs, metric direction, controls and caveats. Paste the actual response into your evidence sheet. The following intentionally wrong recommendation is a deterministic test case, not a claimed live assistant call.

In [4]:
wrong=rows.sort_values('rmse_kwh').iloc[-1]
print(verify_summary(rows,wrong.run_id))
print(verify_summary(rows,best.run_id))
assert not verify_summary(rows,wrong.run_id)['recommendation_matches_metric']
assert verify_summary(rows,best.run_id)['recommendation_matches_metric']

{'recommended_run_id': '069962b2a74647f6a857f94dbcb9e373', 'best_run_id': '67dd70d163a24d3e8e4202b0917ae7ce', 'best_rmse_kwh': 1.4342817684933908, 'recommendation_matches_metric': False}
{'recommended_run_id': '67dd70d163a24d3e8e4202b0917ae7ce', 'best_run_id': '67dd70d163a24d3e8e4202b0917ae7ce', 'best_rmse_kwh': 1.4342817684933908, 'recommendation_matches_metric': True}


## Prepare a UI comparison
For this UI task, use the local setup: run the printed command in a module-a terminal and open localhost:5050. In hosted Colab, prepare a local matrix instead; a hosted localhost address is not a browser connection. In `linden-demand`, select the five rows, compare parameters and validation metrics, and sort RMSE ascending. Confirm the displayed run ID matches the raw export. API evidence does not substitute for doing this UI task.

In [5]:
print('campus-mlops ui --state',ws.path)
print('Best source run:',best.run_id)
print('Raw CSV:',ws.path/'runs.csv')

campus-mlops ui --state /private/var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-vruy7dfn/tracking
Best source run: 67dd70d163a24d3e8e4202b0917ae7ce
Raw CSV: /private/var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-vruy7dfn/tracking/runs.csv
